# From data to geometry

Run each cell in order to build a live 3D bar chart inside Blender — no files to export, no extra setup needed.

In [ ]:
import bpy
import numpy as np
import ipywidgets as widgets
from IPython.display import display

print("Blender", bpy.app.version_string)

In [9]:
COLLECTION = "DataBars"


def fresh_scene():
    """Empty the scene, then add a framed camera and a sun."""
    bpy.ops.object.select_all(action="SELECT")
    bpy.ops.object.delete()

    cam_data = bpy.data.cameras.new("Camera")
    cam = bpy.data.objects.new("Camera", cam_data)
    cam.location = (0.0, -22.0, 9.0)
    cam.rotation_euler = (np.radians(72), 0.0, 0.0)
    bpy.context.scene.collection.objects.link(cam)
    bpy.context.scene.camera = cam

    sun_data = bpy.data.lights.new("Sun", type="SUN")
    sun_data.energy = 4.0
    sun = bpy.data.objects.new("Sun", sun_data)
    sun.rotation_euler = (np.radians(35), np.radians(15), np.radians(20))
    bpy.context.scene.collection.objects.link(sun)


def render():
    """Render the active camera and display the result inline."""
    scene = bpy.context.scene
    scene.render.engine = "BLENDER_EEVEE"
    scene.render.resolution_x = 640
    scene.render.resolution_y = 360
    bpy.ops.render.render()
    path = os.path.join(tempfile.gettempdir(), "jb_render.png")
    bpy.data.images["Render Result"].save_render(filepath=path)
    display(Image(filename=path))

In [10]:
months = np.array(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
)
rainfall = np.array(
    [62, 47, 54, 41, 78, 99, 121, 110, 84, 67, 58, 70], dtype=float
)

# Quick text preview, no plotting library needed.
for name, value in zip(months, rainfall):
    bar = "#" * int(value / 4)
    print(f"{name}  {bar} {value:.0f} mm")

Jan  ############### 62 mm
Feb  ########### 47 mm
Mar  ############# 54 mm
Apr  ########## 41 mm
May  ################### 78 mm
Jun  ######################## 99 mm
Jul  ############################## 121 mm
Aug  ########################### 110 mm
Sep  ##################### 84 mm
Oct  ################ 67 mm
Nov  ############## 58 mm
Dec  ################# 70 mm


## Build bars

In [ ]:
months = np.array(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
)
rainfall = np.array(
    [62, 47, 54, 41, 78, 99, 121, 110, 84, 67, 58, 70], dtype=float
)

## Color by value

In [ ]:
def value_to_color(t):
    return (t, 0.15, 1.0 - t, 1.0)

span = rainfall.max() - rainfall.min()
norm = (rainfall - rainfall.min()) / span
for bar, t in zip(bars, norm):
    mat = bpy.data.materials.new(name=f"mat_{bar.name}")
    mat.use_nodes = True
    bsdf = mat.node_tree.nodes["Principled BSDF"]
    bsdf.inputs["Base Color"].default_value = value_to_color(float(t))
    bar.data.materials.clear()
    bar.data.materials.append(mat)

## Live reshape

Drag the slider to scale all bars in the viewport in real time.

In [13]:
exaggeration = widgets.FloatSlider(
    value=1.0, min=0.2, max=3.0, step=0.1, description="scale"
)
base_heights = heights.copy()


def reshape(_change=None):
    factor = exaggeration.value
    for bar, h in zip(bars, base_heights):
        new_h = float(h * factor)
        bar.scale.z = new_h
        bar.location.z = new_h / 2.0
    bpy.context.view_layer.update()


exaggeration.observe(reshape, names="value")
display(exaggeration)

FloatSlider(value=1.0, description='scale', max=3.0, min=0.2)